In [ ]:
from preamble_jax import *
from scipy.optimize import curve_fit

In [ ]:
import numpy as np
from scipy.signal import find_peaks
from typing import Tuple, List, Optional
import matplotlib.pyplot as plt

def simple_bin_optimizer(visit: str = 'F21', 
                        min_bins: int = 2, 
                        max_bins: int = 20,
                        initial_bins: Optional[int] = None,
                        refinement_steps: int = 3,
                        plot_progress: bool = True) -> Tuple[np.ndarray, float, dict]:
    """
    Simple greedy algorithm to find good bin edges with specified bin count range.
    
    Parameters:
    -----------
    visit : str
        Visit identifier
    min_bins : int
        Minimum number of bins to test
    max_bins : int
        Maximum number of bins to test
    initial_bins : int, optional
        Starting number of bins (defaults to min_bins)
    refinement_steps : int
        Number of refinement iterations after initial placement
    plot_progress : bool
        Whether to plot optimization progress
        
    Returns:
    --------
    Tuple containing:
    - best_edges : np.ndarray
        Optimized bin edges
    - best_rms : float
        Best average RMS achieved
    - history : dict
        Optimization history for analysis
    """
    
    # Initialize history tracking
    history = {
        'n_bins_tested': [],
        'rms_values': [],
        'edges_by_nbins': {},
        'rms_by_nbins': {},
        'wavelengths': None,
        'rms_per_wavelength': None
    }
    
    print(f"Starting greedy bin optimization for {visit}")
    print(f"Testing from {min_bins} to {max_bins} bins")
    
    # Get data once to establish baseline
    print("\n1. Getting baseline data and RMS per wavelength...")
    rms_temp, wavelength_range, wavelengths = bin_testing(
        visit=visit,
        bin_edges=np.linspace(0.82,1.13,108),
        print_results=False,
        plot=False,
        return_wavelengths=True
    )
    
    wmin, wmax = wavelength_range
    n_wavelengths = len(wavelengths)
    
    # Store for history
    history['wavelengths'] = wavelengths
    history['rms_per_wavelength'] = rms_temp
    
    # Calculate baseline (single bin)
    baseline_rms = np.mean(rms_temp)
    print(f"   Baseline (single bin) RMS: {baseline_rms:.2f}")
    
    # Smooth RMS curve for better minima detection
    window_size = min(7, n_wavelengths // 10)
    if window_size % 2 == 0:
        window_size += 1
    rms_smooth = np.convolve(rms_temp, np.ones(window_size)/window_size, mode='same')
    
    # Step 1: Find candidate split points from RMS profile
    print("\n2. Analyzing RMS profile to find candidate split points...")
    
    # Find local minima in RMS
    minima_idx, minima_props = find_peaks(-rms_smooth, 
                                         distance=5,  # Minimum distance between minima
                                         prominence=0.1*np.std(rms_smooth))  # Minimum prominence
    
    minima_wavelengths = wavelengths[minima_idx]
    minima_rms_values = rms_smooth[minima_idx]
    
    # Sort minima by depth (lowest RMS first)
    sorted_indices = np.argsort(minima_rms_values)
    candidate_splits = minima_wavelengths[sorted_indices]
    
    print(f"   Found {len(candidate_splits)} candidate split points")
    
    if plot_progress:
        fig, axes = plt.subplots(2, 1, figsize=(10, 8))
        axes[0].plot(wavelengths, rms_temp, 'k-', alpha=0.3, label='Raw RMS')
        axes[0].plot(wavelengths, rms_smooth, 'b-', linewidth=2, label='Smoothed RMS')
        axes[0].scatter(minima_wavelengths, minima_rms_values, 
                       color='red', s=50, zorder=5, label='Local minima')
        axes[0].set_xlabel('Wavelength')
        axes[0].set_ylabel('RMS')
        axes[0].set_title(f'{visit}: RMS Profile and Candidate Split Points')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    # Step 2: Evaluate different numbers of bins
    print("\n3. Testing different numbers of bins...")
    
    best_rms = baseline_rms
    best_edges = np.array([wmin, wmax])
    best_n_bins = 1
    
    # If initial_bins not specified, start from min_bins
    if initial_bins is None:
        initial_bins = min_bins
    
    # Test from initial_bins up to max_bins
    for n_bins in range(initial_bins, max_bins + 1):
        print(f"   Testing {n_bins} bins...", end=' ')
        
        # Determine how many split points to use
        n_splits_needed = n_bins - 1
        
        if len(candidate_splits) >= n_splits_needed:
            # Use the best (deepest) minima as split points
            splits = candidate_splits[:n_splits_needed]
        else:
            # Not enough minima found - create evenly spaced splits
            splits = np.linspace(wmin + 0.1*(wmax-wmin), 
                               wmax - 0.1*(wmax-wmin), 
                               n_splits_needed)
            print(f"(using evenly spaced - only {len(candidate_splits)} minima found)", end=' ')
        
        # Create edges
        edges = np.sort(np.concatenate([[wmin], splits, [wmax]]))
        
        # Refine edges through iterative improvement
        current_edges = edges.copy()
        current_rms = float('inf')
        
        for refine_step in range(refinement_steps):
            # Evaluate current edges
            rms_list = bin_testing(
                visit=visit,
                bin_edges=current_edges,
                print_results=False,
                plot=False
            )
            avg_rms = np.mean(rms_list)
            
            # Try to improve each inner edge
            improved_edges = current_edges.copy()
            
            for i in range(1, len(current_edges) - 1):
                # Try moving this edge left and right
                left_boundary = current_edges[i-1] + 0.01*(wmax-wmin)
                right_boundary = current_edges[i+1] - 0.01*(wmax-wmin)
                
                # Test a few positions around current edge
                test_positions = np.linspace(left_boundary, right_boundary, 5)
                best_pos = current_edges[i]
                best_edge_rms = avg_rms
                
                for pos in test_positions:
                    temp_edges = current_edges.copy()
                    temp_edges[i] = pos
                    
                    # Keep edges sorted
                    temp_edges.sort()
                    
                    # Quick evaluation (could use interpolation instead of full run)
                    # For simplicity, we'll do full evaluation
                    temp_rms_list = bin_testing(
                        visit=visit,
                        bin_edges=temp_edges,
                        print_results=False,
                        plot=False
                    )
                    temp_avg_rms = np.mean(temp_rms_list)
                    
                    if temp_avg_rms < best_edge_rms:
                        best_edge_rms = temp_avg_rms
                        best_pos = pos
                
                improved_edges[i] = best_pos
            
            # Update current edges
            current_edges = improved_edges.copy()
            current_rms = best_edge_rms
        
        # Final evaluation
        final_rms_list = bin_testing(
            visit=visit,
            bin_edges=current_edges,
            print_results=False,
            plot=False
        )
        final_avg_rms = np.mean(final_rms_list)
        
        # Store in history
        history['n_bins_tested'].append(n_bins)
        history['rms_values'].append(final_avg_rms)
        history['edges_by_nbins'][n_bins] = current_edges
        history['rms_by_nbins'][n_bins] = final_avg_rms
        
        print(f"RMS: {final_avg_rms:.2f}")
        
        # Update best configuration
        if final_avg_rms < best_rms:
            best_rms = final_avg_rms
            best_edges = current_edges
            best_n_bins = n_bins
    
    # Step 3: Optional downward search from best_n_bins to min_bins
    if best_n_bins > min_bins:
        print(f"\n4. Refining downward from {best_n_bins} to {min_bins} bins...")
        
        # Start from best configuration and remove bins
        current_edges = best_edges.copy()
        current_n_bins = best_n_bins
        current_rms = best_rms
        
        while current_n_bins > min_bins:
            print(f"   Testing {current_n_bins-1} bins (by removing one)...", end=' ')
            
            # Find which edge to remove (one with smallest impact)
            best_removed_rms = float('inf')
            best_removed_edges = None
            
            # Try removing each inner edge
            for i in range(1, len(current_edges) - 1):
                # Create edges without this split point
                edges_without = np.delete(current_edges, i)
                
                # Evaluate
                rms_list = bin_testing(
                    visit=visit,
                    bin_edges=edges_without,
                    print_results=False,
                    plot=False
                )
                avg_rms = np.mean(rms_list)
                
                if avg_rms < best_removed_rms:
                    best_removed_rms = avg_rms
                    best_removed_edges = edges_without
            
            if best_removed_rms < best_rms:
                # Actually better with fewer bins
                best_rms = best_removed_rms
                best_edges = best_removed_edges
                best_n_bins = current_n_bins - 1
                
                # Store in history
                if best_n_bins not in history['edges_by_nbins']:
                    history['edges_by_nbins'][best_n_bins] = best_edges
                    history['rms_by_nbins'][best_n_bins] = best_rms
                    history['n_bins_tested'].append(best_n_bins)
                    history['rms_values'].append(best_rms)
                
                print(f"RMS: {best_removed_rms:.2f} (NEW BEST!)")
            else:
                print(f"RMS: {best_removed_rms:.2f} (worse)")
                break  # Stop if removing bins makes things worse
            
            current_edges = best_edges
            current_n_bins = best_n_bins
            current_rms = best_rms
    
    # Final results
    print(f"\n{'='*60}")
    print(f"OPTIMIZATION COMPLETE")
    print(f"{'='*60}")
    print(f"Best configuration: {best_n_bins} bins")
    print(f"Best RMS: {best_rms:.2f} (improvement of {baseline_rms-best_rms:.2f} from baseline)")
    print(f"Bin edges: {best_edges}")
    print(f"{'='*60}")
    
    # Plot optimization history if requested
    if plot_progress:
        # Sort history for plotting
        sorted_indices = np.argsort(history['n_bins_tested'])
        n_bins_sorted = np.array(history['n_bins_tested'])[sorted_indices]
        rms_sorted = np.array(history['rms_values'])[sorted_indices]
        
        axes[1].plot(n_bins_sorted, rms_sorted, 'bo-', linewidth=2, markersize=8)
        axes[1].axhline(y=baseline_rms, color='r', linestyle='--', label=f'Baseline (1 bin): {baseline_rms:.2f}')
        axes[1].axvline(x=best_n_bins, color='g', linestyle=':', label=f'Best: {best_n_bins} bins')
        axes[1].scatter(best_n_bins, best_rms, color='green', s=100, zorder=5)
        
        axes[1].set_xlabel('Number of Bins')
        axes[1].set_ylabel('Average RMS')
        axes[1].set_title('Optimization Progress')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'../../figs/{visit}_bin_optimization_progress.png', dpi=300)
        plt.show()
        
        # Also plot the final best bin configuration
        fig2, ax2 = plt.subplots(figsize=(12, 6))
        
        # Plot RMS per wavelength with final bins
        ax2.plot(wavelengths, rms_temp, 'k-', alpha=0.5, label='RMS per wavelength')
        
        # Add bin edges
        for edge in best_edges:
            ax2.axvline(x=edge, color='r', linestyle='--', alpha=0.7)
        
        # Shade bin regions
        for i in range(len(best_edges) - 1):
            ax2.axvspan(best_edges[i], best_edges[i+1], alpha=0.1, color='blue')
        
        ax2.set_xlabel('Wavelength')
        ax2.set_ylabel('RMS')
        ax2.set_title(f'{visit}: Final Optimized Bins ({best_n_bins} bins, RMS={best_rms:.2f})')
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'../../figs/{visit}_final_optimized_bins.png', dpi=300)
        plt.show()
    
    return best_edges, best_rms, history


# Alternative version with more aggressive optimization
def advanced_greedy_optimizer(visit: str = 'F21',
                            min_bins: int = 2,
                            max_bins: int = 20,
                            n_random_starts: int = 5,
                            convergence_threshold: float = 0.01,
                            max_iterations: int = 100) -> Tuple[np.ndarray, float, dict]:
    """
    More advanced greedy optimizer with multiple random restarts.
    
    Parameters:
    -----------
    n_random_starts : int
        Number of random initializations to try
    convergence_threshold : float
        RMS improvement threshold for convergence
    max_iterations : int
        Maximum iterations per bin count
    """
    
    print(f"Advanced greedy optimization for {visit}")
    print(f"Testing {min_bins}-{max_bins} bins with {n_random_starts} random starts each")
    

    
    wmin, wmax = wavelength_range
    
    # Initialize results
    best_overall_rms = float('inf')
    best_overall_edges = None
    best_overall_n_bins = 0
    
    all_results = []
    
    # Test each bin count
    for n_bins in range(min_bins, max_bins + 1):
        print(f"\nTesting {n_bins} bins...")
        
        best_rms_for_n = float('inf')
        best_edges_for_n = None
        
        # Try multiple random starting configurations
        for start_idx in range(n_random_starts):
            print(f"  Random start {start_idx+1}/{n_random_starts}...", end=' ')
            
            # Initialize random edges
            if start_idx == 0:
                # First try: evenly spaced
                edges = np.linspace(wmin, wmax, n_bins + 1)
            else:
                # Random edges
                inner_edges = np.random.uniform(wmin, wmax, n_bins - 1)
                inner_edges.sort()
                edges = np.concatenate([[wmin], inner_edges, [wmax]])
            
            # Local optimization loop
            iteration = 0
            prev_rms = float('inf')
            converged = False
            
            while not converged and iteration < max_iterations:
                # Evaluate current edges
                rms_list = bin_testing(
                    visit=visit,
                    bin_edges=edges,
                    print_results=False,
                    plot=False
                )
                current_rms = np.mean(rms_list)
                
                # Check convergence
                if iteration > 0 and abs(prev_rms - current_rms) < convergence_threshold:
                    converged = True
                
                prev_rms = current_rms
                
                # Try to optimize each edge
                for i in range(1, len(edges) - 1):
                    # Define search range for this edge
                    left = edges[i-1] + 0.001 * (wmax - wmin)
                    right = edges[i+1] - 0.001 * (wmax - wmin)
                    
                    # Sample points around current edge
                    test_points = np.linspace(left, right, 7)
                    best_point = edges[i]
                    best_point_rms = current_rms
                    
                    for point in test_points:
                        test_edges = edges.copy()
                        test_edges[i] = point
                        test_edges.sort()
                        
                        test_rms_list = bin_testing(
                            visit=visit,
                            bin_edges=test_edges,
                            print_results=False,
                            plot=False
                        )
                        test_rms = np.mean(test_rms_list)
                        
                        if test_rms < best_point_rms:
                            best_point_rms = test_rms
                            best_point = point
                    
                    edges[i] = best_point
                
                iteration += 1
            
            # Final evaluation
            final_rms_list = bin_testing(
                visit=visit,
                bin_edges=edges,
                print_results=False,
                plot=False
            )
            final_rms = np.mean(final_rms_list)
            
            print(f"RMS: {final_rms:.2f}")
            
            if final_rms < best_rms_for_n:
                best_rms_for_n = final_rms
                best_edges_for_n = edges.copy()
        
        # Store best for this n_bins
        all_results.append({
            'n_bins': n_bins,
            'rms': best_rms_for_n,
            'edges': best_edges_for_n
        })
        
        print(f"  Best for {n_bins} bins: RMS = {best_rms_for_n:.2f}")
        
        # Update overall best
        if best_rms_for_n < best_overall_rms:
            best_overall_rms = best_rms_for_n
            best_overall_edges = best_edges_for_n
            best_overall_n_bins = n_bins
    
    # Print summary
    print(f"\n{'='*60}")
    print(f"FINAL RESULTS")
    print(f"{'='*60}")
    
    for result in all_results:
        marker = " *" if result['n_bins'] == best_overall_n_bins else ""
        print(f"{result['n_bins']:2d} bins: RMS = {result['rms']:6.2f}{marker}")
    
    print(f"\nBest: {best_overall_n_bins} bins with RMS = {best_overall_rms:.2f}")
    print(f"Bin edges: {best_overall_edges}")
    
    return best_overall_edges, best_overall_rms, {'all_results': all_results}


# Convenience function for common use cases
def optimize_bins(visit: str = 'F21',
                 method: str = 'simple',
                 min_bins: int = 2,
                 max_bins: int = 15,
                 **kwargs) -> np.ndarray:
    """
    High-level function to optimize bins with chosen method.
    
    Parameters:
    -----------
    method : str
        'simple' or 'advanced'
    **kwargs : additional arguments passed to the optimizer
    """
    
    if method.lower() == 'simple':
        best_edges, best_rms, history = simple_bin_optimizer(
            visit=visit,
            min_bins=min_bins,
            max_bins=max_bins,
            **kwargs
        )
    elif method.lower() == 'advanced':
        best_edges, best_rms, history = advanced_greedy_optimizer(
            visit=visit,
            min_bins=min_bins,
            max_bins=max_bins,
            **kwargs
        )
    else:
        raise ValueError(f"Unknown method: {method}. Choose 'simple' or 'advanced'.")
    
    return best_edges

In [ ]:
def bin_testing(visit='F21',bin_edges = None,
                print_results='True',plot='True',return_wavelengths=False):

    predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
    binwidth = visits[f'{visit}']['native resolution']
    exptime = visits[f'{visit}']['exp (s)']
    grism = visits[f'{visit}']['Grism']
    
    rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
    # print(f'N_wavelengths in visit {visit} = {rainbow.nwave}')
    # print(f'N_times in visit {visit} = {rainbow.ntime}')
    wavelength_range = [rainbow.wavelength.value.min(), rainbow.wavelength.value.max()]
    
    data_wavelengths = rainbow.wavelength.value
    img_date = rainbow.time.value
    _data_flux = rainbow.flux.value
    time_from_T0 = img_date - predicted_T0

    mean_data_flux = np.nanmean(_data_flux, axis=1)
    data_flux = _data_flux / mean_data_flux[:, np.newaxis]
    relative_err = rainbow.uncertainty.value/_data_flux

    # Label the orbits
    orbit = np.zeros_like(img_date)
    for j in range(len(img_date)):
        if j >= 1:
            if (img_date[j] - img_date[j - 1]) > 0.01:
                orbit[j] = (orbit[j - 1] + 1)
            else:
                orbit[j] = orbit[j - 1]
    
    # Trim the first point from each orbit
    ref_time = []
    for o in np.unique(orbit):
        first_index = np.where(orbit == o)[0][0]
        ref_time.append(img_date[first_index])
        data_flux[:, first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
        relative_err[:, first_index] = np.nan
        img_date[first_index] = np.nan
        time_from_T0[first_index] = np.nan
    
    # Set data to nan if it was in the pre-defined list of orbits to exclude
    for orbit_to_exclude in np.array([0,2,3,4,5]):
        data_flux[:, orbit == orbit_to_exclude] = np.nan
        relative_err[:, orbit == orbit_to_exclude] = np.nan
        img_date[orbit == orbit_to_exclude] = np.nan
        time_from_T0[orbit == orbit_to_exclude] = np.nan
    
    # Populate ramp_phase time arrays
    phase_list=[]
    for o in [0,1,2,3,4,5,6,7]:
        rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
        phase_list.append(rphase)
    ramp_phase = np.concatenate(phase_list)
    
    nanmask = ~np.isnan(img_date)
    data_flux = data_flux[:,nanmask]
    relative_err = relative_err[:,nanmask]
    time_from_T0 = time_from_T0[nanmask]  
    img_date = img_date[nanmask]          
    ramp_phase = ramp_phase[nanmask]
    
    def linear_model(x, m, b):
    
        ramp = ramp_model(phase=ramp_phase,r1=6,r2=-6.0,r3=0.0)
    
        return (m * x + b)*ramp
    
    # Residual function (for minimization)
    def residual(params, x, y):
        m, b = params
        model = linear_model(x, m, b)
        return y - model # Residual = data - model
        
    'PLOT EACH LIGHT CURVE WITH THE MODEL, NEXT TO THE RESIDUALS'
    # fig, axes = plt.subplots(1,2,figsize=(10, 1.5))
    popt_list = []
    initial_guess = [-0.03, 1.0]  # m, b
    for i in range(len(data_wavelengths)):
        # for ax in axes:
        #     ax.clear()
        
        time = time_from_T0
        flux = data_flux[i, :]
        err = relative_err[i, :]
    
        # Fit using curve_fit
        popt, pcov = curve_fit(
            lambda x, m, b: linear_model(x, m, b),
            time,
            flux,
            sigma = err,
            p0=initial_guess,
            maxfev = 100000
        )
        # Best-fit parameters
        popt_list.append(popt)
        m_opt, b_opt = popt
        # print("Best-fit params:", popt)
        model_flux = linear_model(time_from_T0, m_opt, b_opt)
        residual = data_flux[i,:]-model_flux
        rms = int(np.nanstd(residual)*1e6)
        chisq = np.nansum((residual/relative_err[i,:])**2)
        rchisq = chisq/(np.sum(~np.isnan(time_from_T0))-2)
            
        if print_results:
            if i == 0:
                print('Wavelength Median_rel_err linear_rms r_chisq')
            print(f'{data_wavelengths[i]:.5f} {int(np.nanmedian(relative_err[i,:])*1e6)} {rms} {rchisq:.3f}')
        # axes[0].set_title(f'{data_wavelengths[i]:.5f} micron')
        # axes[1].set_title(f'Residual RMS = {rms} ppm')
        # axes[0].errorbar(img_date, data_flux[i,:],yerr=relative_err[i,:],fmt='o',
        #                  ms=1, label = f'Reduced chi-squared = {rchisq:.3f}',color='b')
        # axes[0].legend(loc = 'upper right')
        # axes[0].plot(img_date, model_flux,color='r')
        # axes[1].scatter(img_date, data_flux[i,:]-model_flux,s=1)
        # plt.savefig(f'../figs/{visit}_unbinned_lc_{i}.png')
    
    'RMS and Bin Edge Plots'
    if plot:
        fig, axes = plt.subplots(4,1,figsize=(8.0, 9),sharex=True)
        axes[0].set_title(f'{visit} N_bins = {len(bin_edges)-1}')
        # axes[0].set_title(f'Data - Model (slope x ramp)')
        axes[3].set_xlabel(r'Wavelength $\mu$m')
        axes[0].set_ylabel(f'Residual RMS (ppm)')
        axes[1].set_ylabel(r'Reduced $\chi^2$')
        axes[2].set_ylabel(f'Residual RMS (ppm)')
        axes[3].set_ylabel(r'Reduced $\chi^2$')
    rms_list = []
    for i in range(len(data_wavelengths)):
        m_opt, b_opt = popt_list[i]
        model_flux = linear_model(time_from_T0, m_opt, b_opt)
        residual = data_flux[i,:]-model_flux
        rms = int(np.nanstd(residual)*1e6)
        rms_list.append(rms )
        chisq = np.nansum((residual/relative_err[i,:])**2)
        rchisq = chisq/(np.sum(~np.isnan(time_from_T0))-2)
        if plot:
            axes[0].scatter(data_wavelengths[i], rms,color='pink', zorder=100,alpha=0.5)
            axes[1].scatter(data_wavelengths[i], rchisq, color='k', zorder=100)

    if plot:
        for edge in bin_edges:
            axes[0].axvline(edge,color='r', linewidth=1)
            axes[1].axvline(edge,color='r', linewidth=1)
            axes[2].axvline(edge,color='r', linewidth=1)
            axes[3].axvline(edge,color='r', linewidth=1)
        axes[0].plot(data_wavelengths, rms_list, color='k',lw=1)

    # Now do this again but with a binnned light curve

    predicted_T0 = visits[f'{visit}']['T0 (BJD_TDB)'].value
    binwidth = visits[f'{visit}']['native resolution']
    exptime = visits[f'{visit}']['exp (s)']
    grism = visits[f'{visit}']['Grism']
    
    rainbow = read_rainbow(f"../../data/rainbows/{visit}_scan-combined_trimmed_pacman_spec.rainbow.npy")
    rainbow = rainbow.bin(wavelength_edges=bin_edges*u.micron,minimum_points_per_bin=1)
    
    data_wavelengths = rainbow.wavelength.value
    img_date = rainbow.time.value
    _data_flux = rainbow.flux.value
    relative_err = rainbow.uncertainty.value/_data_flux
    time_from_T0 = img_date - predicted_T0

    mean_data_flux = np.nanmean(_data_flux, axis=1)
    data_flux = _data_flux / mean_data_flux[:, np.newaxis]
    
    # Label the orbits
    orbit = np.zeros_like(img_date)
    for j in range(len(img_date)):
        if j >= 1:
            if (img_date[j] - img_date[j - 1]) > 0.01:
                orbit[j] = (orbit[j - 1] + 1)
            else:
                orbit[j] = orbit[j - 1]
    
    # Trim the first point from each orbit
    ref_time = []
    for o in np.unique(orbit):
        first_index = np.where(orbit == o)[0][0]
        ref_time.append(img_date[first_index])
        data_flux[:, first_index] = np.nan  # Set the first point of each subsequent orbit to np.nan
        relative_err[:, first_index] = np.nan
        img_date[first_index] = np.nan
        time_from_T0[first_index] = np.nan
    
    # Set data to nan if it was in the pre-defined list of orbits to exclude
    for orbit_to_exclude in np.array([0,2,3,4,5]):
        data_flux[:, orbit == orbit_to_exclude] = np.nan
        relative_err[:, orbit == orbit_to_exclude] = np.nan
        img_date[orbit == orbit_to_exclude] = np.nan
        time_from_T0[orbit == orbit_to_exclude] = np.nan
    
    # Populate ramp_phase time arrays
    phase_list=[]
    for o in [0,1,2,3,4,5,6,7]:
        rphase = (img_date[orbit==o] - ref_time[o]) / 0.066
        phase_list.append(rphase)
    ramp_phase = np.concatenate(phase_list)
    
    nanmask = ~np.isnan(img_date)
    data_flux = data_flux[:,nanmask]
    relative_err = relative_err[:,nanmask]
    time_from_T0 = time_from_T0[nanmask]  
    img_date = img_date[nanmask]          
    ramp_phase = ramp_phase[nanmask]
        
    'PLOT EACH LIGHT CURVE WITH THE MODEL, NEXT TO THE RESIDUALS'
    # lc_fig, lc_axes = plt.subplots(1,2,figsize=(10, 1.5))
    popt_list = []
    for i in range(len(data_wavelengths)):
        # for ax in axes:
        #     ax.clear()
        
        time = time_from_T0
        flux = data_flux[i, :]
        err = relative_err[i, :]
    
        # Fit using curve_fit
        popt, pcov = curve_fit(
            lambda x, m, b: linear_model(x, m, b),
            time,
            flux,
            sigma = err,
            p0=initial_guess,
            maxfev = 100000
        )
        # Best-fit parameters
        popt_list.append(popt)
        m_opt, b_opt = popt
        model_flux = linear_model(time_from_T0, m_opt, b_opt)
        residual = data_flux[i,:]-model_flux
        rms = int(np.nanstd(residual)*1e6)
        chisq = np.nansum((residual/relative_err[i,:])**2)
        rchisq = chisq/(np.sum(~np.isnan(time_from_T0))-2)
        if print_results:
            if i == 0:
                print('BINNED: Wavelength Median_rel_err linear_rms r_chisq')
            print(f'{data_wavelengths[i]:.5f} {int(np.nanmedian(relative_err[i,:])*1e6)} {rms} {rchisq:.3f}')
        # lc_axes[0].set_title(f'{data_wavelengths[i]:.5f} micron')
        # lc_axes[1].set_title(f'Residual RMS = {rms} ppm')
        # lc_axes[0].errorbar(img_date, data_flux[i,:],yerr=relative_err[i,:],fmt='o',
        #                  ms=1, label = f'Reduced chi-squared = {rchisq:.3f}',color='b')
        # lc_axes[0].legend(loc = 'upper right')
        # lc_axes[0].plot(img_date, model_flux,color='r')
        # lc_axes[1].scatter(img_date, data_flux[i,:]-model_flux,s=1)
        # plt.savefig(f'../figs/{visit}_binned_lc_{i}.png')
    
    'PLOT THE RMS'
    rms_list = []
    for i in range(len(data_wavelengths)):
        m_opt, b_opt = popt_list[i]
        model_flux = linear_model(time_from_T0, m_opt, b_opt)
        residual = data_flux[i,:]-model_flux
        rms = int(np.nanstd(residual)*1e6)
        rms_list.append( rms )
        chisq = np.nansum((residual/relative_err[i,:])**2)
        rchisq = chisq/(np.sum(~np.isnan(time_from_T0))-2)
        if plot:
            axes[2].scatter(data_wavelengths[i], rms, color='pink', zorder=100,alpha=0.5)
            axes[3].scatter(data_wavelengths[i], rchisq, color='k', zorder=100)
            
    if plot:
        axes[2].plot(data_wavelengths, rms_list, color='k')
        # axes[0].set_ylim(400,None)
        axes[2].set_ylim(250,1000)
        # axes[3].set_ylim(0.5,30)
        # axes[1].set_yscale('log')
        axes[2].set_yscale('log')
        axes[3].set_yscale('log')

        plt.savefig(f'../../figs/{visit}_{len(bin_edges)-1}bins_rms_plots.png',dpi=400)
        plt.show()

    if return_wavelengths:
        # Return both RMS list and wavelength info
        return rms_list, wavelength_range, data_wavelengths
    else:
        return rms_list

In [ ]:
def robust_bin_optimizer(visit: str = 'F21', 
                        min_bins: int = 16, 
                        max_bins: int = 60,
                        force_even_spacing: bool = False,
                        adaptive_smoothing: bool = True,
                        plot_progress: bool = True) -> Tuple[np.ndarray, float, dict]:
    """
    More robust version that handles cases where no minima are found.
    
    Parameters:
    -----------
    force_even_spacing : bool
        If True, force evenly spaced bins regardless of RMS profile
    adaptive_smoothing : bool
        If True, adapt smoothing based on data characteristics
    """
    
    # Initialize history
    history = {
        'n_bins_tested': [],
        'rms_values': [],
        'edges_by_nbins': {},
        'rms_by_nbins': {},
        'wavelengths': None,
        'rms_per_wavelength': None,
        'method_used': []
    }
    
    print(f"Starting robust bin optimization for {visit}")
    print(f"Testing from {min_bins} to {max_bins} bins")
    
    # Get data once
    print("\n1. Getting baseline data and RMS per wavelength...")
    rms_temp, wavelength_range, wavelengths = bin_testing(
        visit=visit,
        bin_edges=np.linspace(0.82, 1.13, 108),
        print_results=False,
        plot=False,
        return_wavelengths=True
    )
    
    wmin, wmax = wavelength_range
    n_wavelengths = len(wavelengths)
    
    # Store for history
    history['wavelengths'] = wavelengths
    history['rms_per_wavelength'] = rms_temp
    
    baseline_rms = np.mean(rms_temp)
    print(f"   Baseline (single bin) RMS: {baseline_rms:.2f}")
    print(f"   Number of wavelength points: {n_wavelengths}")
    
    # Analyze RMS statistics
    rms_mean = np.mean(rms_temp)
    rms_std = np.std(rms_temp)
    rms_range = np.max(rms_temp) - np.min(rms_temp)
    
    print(f"   RMS statistics - Mean: {rms_mean:.2f}, Std: {rms_std:.2f}, Range: {rms_range:.2f}")
    
    # Step 1: Find candidate split points using multiple methods
    print("\n2. Finding candidate split points...")
    
    candidate_methods = []
    candidate_splits = []
    
    # Method 1: Try different smoothing levels
    if adaptive_smoothing:
        smoothing_levels = [3, 5, 7, 9, 11, 15]
    else:
        smoothing_levels = [5]
    
    for window in smoothing_levels:
        if window < n_wavelengths:
            # Apply smoothing
            rms_smooth = np.convolve(rms_temp, np.ones(window)/window, mode='same')
            
            # Try different prominence thresholds
            for prominence_factor in [0.05, 0.1, 0.2, 0.3]:
                prominence_threshold = prominence_factor * rms_std
                
                # Find minima
                try:
                    minima_idx, minima_props = find_peaks(
                        -rms_smooth,
                        distance=max(3, n_wavelengths // 50),  # Adaptive distance
                        prominence=prominence_threshold
                    )
                    
                    if len(minima_idx) > 0:
                        minima_wavelengths = wavelengths[minima_idx]
                        candidate_splits.extend(minima_wavelengths.tolist())
                        candidate_methods.append(f"window={window}, prom={prominence_factor}")
                        
                except Exception as e:
                    continue
    
    # Method 2: Use percentiles of RMS distribution
    if len(candidate_splits) == 0:
        print("   Using percentile-based method...")
        
        # Find wavelengths where RMS is below certain percentiles
        for percentile in [10, 20, 30, 40, 50, 60, 70]:
            threshold = np.percentile(rms_temp, percentile)
            low_rms_indices = np.where(rms_temp < threshold)[0]
            
            if len(low_rms_indices) > 0:
                # Take median of each contiguous low-RMS region
                regions = []
                current_region = []
                
                for idx in low_rms_indices:
                    if not current_region or idx == current_region[-1] + 1:
                        current_region.append(idx)
                    else:
                        if len(current_region) >= 3:  # Minimum region size
                            regions.append(current_region)
                        current_region = [idx]
                
                if current_region and len(current_region) >= 3:
                    regions.append(current_region)
                
                for region in regions:
                    median_idx = region[len(region) // 2]
                    candidate_splits.append(wavelengths[median_idx])
                    candidate_methods.append(f"percentile={percentile}")
    
    # Method 3: Use gradient changes (alternative to minima)
    if len(candidate_splits) < min_bins and not force_even_spacing:
        print("   Using gradient-based method...")
        
        # Calculate gradient of RMS
        rms_gradient = np.gradient(rms_temp)
        rms_gradient_smooth = np.convolve(rms_gradient, np.ones(5)/5, mode='same')
        
        # Find where gradient changes sign (potential minima/maxima)
        sign_changes = np.where(np.diff(np.sign(rms_gradient_smooth)))[0]
        
        # Filter for reasonable spacing
        filtered_changes = []
        prev_idx = -100  # Initialize with large negative
        
        for idx in sign_changes:
            if idx - prev_idx > n_wavelengths // 100:  # Minimum spacing
                filtered_changes.append(idx)
                prev_idx = idx
        
        if len(filtered_changes) > 0:
            candidate_splits.extend(wavelengths[filtered_changes].tolist())
            candidate_methods.append("gradient_sign_changes")
    
    # Remove duplicates and sort
    if candidate_splits:
        candidate_splits = np.unique(np.sort(candidate_splits))
        print(f"   Found {len(candidate_splits)} candidate split points using methods: {', '.join(set(candidate_methods))}")
    else:
        print("   No candidate split points found with any method")
    
    # Step 2: Generate bin configurations
    print("\n3. Generating and testing bin configurations...")
    
    best_rms = baseline_rms
    best_edges = np.array([wmin, wmax])
    best_n_bins = 1
    
    # Define strategy based on available candidates
    if len(candidate_splits) >= min_bins - 1 or force_even_spacing:
        # We have enough candidates or are forcing even spacing
        for n_bins in range(min_bins, max_bins + 1):
            print(f"   Testing {n_bins} bins...", end=' ')
            
            if force_even_spacing or len(candidate_splits) < n_bins - 1:
                # Use evenly spaced bins
                edges = np.linspace(wmin, wmax, n_bins + 1)
                method_used = "evenly_spaced"
            else:
                # Use candidate splits (choose deepest minima first)
                # Sort candidates by RMS value at that wavelength
                candidate_indices = [np.argmin(np.abs(wavelengths - s)) for s in candidate_splits]
                candidate_rms = [rms_temp[i] for i in candidate_indices]
                
                # Sort by RMS (lowest first)
                sorted_idx = np.argsort(candidate_rms)[:n_bins-1]
                splits = candidate_splits[sorted_idx]
                edges = np.sort(np.concatenate([[wmin], splits, [wmax]]))
                method_used = "rms_minima"
            
            # Evaluate
            rms_list = bin_testing(
                visit=visit,
                bin_edges=edges,
                print_results=False,
                plot=False
            )
            avg_rms = np.mean(rms_list)
            
            # Store in history
            history['n_bins_tested'].append(n_bins)
            history['rms_values'].append(avg_rms)
            history['edges_by_nbins'][n_bins] = edges
            history['rms_by_nbins'][n_bins] = avg_rms
            history['method_used'].append(method_used)
            
            print(f"RMS: {avg_rms:.2f} ({method_used})")
            
            # Update best
            if avg_rms < best_rms:
                improvement = best_rms - avg_rms
                best_rms = avg_rms
                best_edges = edges
                best_n_bins = n_bins
                print(f"      ↑ New best! Improvement: {improvement:.2f}")
    else:
        # Not enough candidates - try a different approach
        print(f"   Not enough candidates ({len(candidate_splits)} found, need at least {min_bins-1})")
        print("   Using hybrid approach...")
        
        # Hybrid: combine candidates with evenly spaced
        for n_bins in range(min_bins, min(max_bins + 1, min_bins + 10)):  # Limit search
            print(f"   Testing {n_bins} bins (hybrid)...", end=' ')
            
            # Start with evenly spaced
            base_edges = np.linspace(wmin, wmax, n_bins + 1)
            
            # If we have some candidates, replace closest edges
            if candidate_splits:
                n_to_replace = min(len(candidate_splits), n_bins - 1)
                
                # For each candidate, replace the nearest evenly-spaced edge
                for i in range(n_to_replace):
                    # Find evenly spaced edge closest to this candidate
                    distances = np.abs(base_edges[1:-1] - candidate_splits[i])
                    closest_idx = np.argmin(distances) + 1  # +1 for inner edges
                    base_edges[closest_idx] = candidate_splits[i]
            
            edges = np.sort(base_edges)
            method_used = "hybrid"
            
            # Evaluate
            rms_list = bin_testing(
                visit=visit,
                bin_edges=edges,
                print_results=False,
                plot=False
            )
            avg_rms = np.mean(rms_list)
            
            # Store
            history['n_bins_tested'].append(n_bins)
            history['rms_values'].append(avg_rms)
            history['edges_by_nbins'][n_bins] = edges
            history['rms_by_nbins'][n_bins] = avg_rms
            history['method_used'].append(method_used)
            
            print(f"RMS: {avg_rms:.2f} ({method_used})")
            
            if avg_rms < best_rms:
                best_rms = avg_rms
                best_edges = edges
                best_n_bins = n_bins
    
    # Step 3: Refinement
    print(f"\n4. Refining best configuration ({best_n_bins} bins)...")
    
    # Try local optimization around best edges
    refinement_iterations = 5
    current_edges = best_edges.copy()
    current_rms = best_rms
    
    for refine_iter in range(refinement_iterations):
        improved = False
        
        # Try adjusting each inner edge
        for i in range(1, len(current_edges) - 1):
            # Define search range
            left_boundary = current_edges[i-1] + 0.005 * (wmax - wmin)
            right_boundary = current_edges[i+1] - 0.005 * (wmax - wmin)
            
            # Test a few positions
            test_positions = np.linspace(left_boundary, right_boundary, 7)
            best_pos = current_edges[i]
            best_pos_rms = current_rms
            
            for pos in test_positions:
                test_edges = current_edges.copy()
                test_edges[i] = pos
                test_edges.sort()
                
                # Quick check if edges are valid
                if np.any(np.diff(test_edges) <= 0):
                    continue
                
                test_rms_list = bin_testing(
                    visit=visit,
                    bin_edges=test_edges,
                    print_results=False,
                    plot=False
                )
                test_rms = np.mean(test_rms_list)
                
                if test_rms < best_pos_rms:
                    best_pos_rms = test_rms
                    best_pos = pos
                    improved = True
            
            current_edges[i] = best_pos
        
        # Update current RMS
        if improved:
            final_check = bin_testing(
                visit=visit,
                bin_edges=current_edges,
                print_results=False,
                plot=False
            )
            current_rms = np.mean(final_check)
            print(f"   Iteration {refine_iter+1}: RMS improved to {current_rms:.2f}")
        else:
            print(f"   Iteration {refine_iter+1}: No improvement")
            break
    
    # Update best if refinement helped
    if current_rms < best_rms:
        best_rms = current_rms
        best_edges = current_edges
        print(f"   Refinement successful! New best RMS: {best_rms:.2f}")
    
    # Final results
    print(f"\n{'='*60}")
    print(f"OPTIMIZATION COMPLETE")
    print(f"{'='*60}")
    print(f"Best configuration: {best_n_bins} bins")
    print(f"Best RMS: {best_rms:.2f} (improvement of {baseline_rms-best_rms:.2f} from baseline)")
    print(f"Number of wavelength points per bin (approx): {n_wavelengths / best_n_bins:.1f}")
    print(f"\nBin edges:")
    for i, edge in enumerate(best_edges):
        if i < len(best_edges) - 1:
            bin_width = best_edges[i+1] - edge
            print(f"  Bin {i+1}: {edge:.5f} to {best_edges[i+1]:.5f} (width: {bin_width:.5f})")
    print(f"{'='*60}")
    
    # Plot results if requested
    if plot_progress:
        plot_optimization_results(visit, wavelengths, rms_temp, best_edges, 
                                best_rms, baseline_rms, history)
    
    return best_edges, best_rms, history


def plot_optimization_results(visit, wavelengths, rms_temp, best_edges, 
                            best_rms, baseline_rms, history):
    """Plot optimization results."""
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: RMS profile with best bins
    ax1 = axes[0, 0]
    ax1.plot(wavelengths, rms_temp, 'k-', alpha=0.7, linewidth=1, label='RMS per λ')
    
    # Add bin edges and shading
    for edge in best_edges:
        ax1.axvline(x=edge, color='r', linestyle='--', alpha=0.7, linewidth=1)
    
    # Shade bins
    for i in range(len(best_edges) - 1):
        ax1.axvspan(best_edges[i], best_edges[i+1], alpha=0.1, color='blue')
    
    ax1.set_xlabel('Wavelength')
    ax1.set_ylabel('RMS (ppm)')
    ax1.set_title(f'{visit}: RMS Profile with Optimized Bins\n({len(best_edges)-1} bins, RMS={best_rms:.2f})')
    ax1.legend(['RMS per λ', 'Bin edges'])
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Optimization progress
    ax2 = axes[0, 1]
    if history['n_bins_tested']:
        # Sort by number of bins
        sorted_indices = np.argsort(history['n_bins_tested'])
        n_bins_sorted = np.array(history['n_bins_tested'])[sorted_indices]
        rms_sorted = np.array(history['rms_values'])[sorted_indices]
        
        ax2.plot(n_bins_sorted, rms_sorted, 'bo-', linewidth=2, markersize=6)
        
        # Color points by method if available
        if 'method_used' in history and len(history['method_used']) == len(n_bins_sorted):
            methods_sorted = np.array(history['method_used'])[sorted_indices]
            
            # Define colors for different methods
            method_colors = {
                'evenly_spaced': 'blue',
                'rms_minima': 'green',
                'hybrid': 'orange',
                'percentile': 'purple',
                'gradient_sign_changes': 'brown'
            }
            
            for i, method in enumerate(methods_sorted):
                color = method_colors.get(method, 'blue')
                ax2.plot(n_bins_sorted[i], rms_sorted[i], 'o', color=color, markersize=8)
    
    ax2.axhline(y=baseline_rms, color='r', linestyle='--', 
                label=f'Baseline (1 bin): {baseline_rms:.2f}')
    ax2.axvline(x=len(best_edges)-1, color='g', linestyle=':', 
                label=f'Best: {len(best_edges)-1} bins')
    
    ax2.set_xlabel('Number of Bins')
    ax2.set_ylabel('Average RMS (ppm)')
    ax2.set_title('Optimization Progress')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Histogram of RMS values
    ax3 = axes[1, 0]
    ax3.hist(rms_temp, bins=30, alpha=0.7, edgecolor='black')
    ax3.axvline(x=baseline_rms, color='r', linestyle='--', label=f'Baseline avg: {baseline_rms:.2f}')
    ax3.axvline(x=best_rms, color='g', linestyle='--', label=f'Best avg: {best_rms:.2f}')
    ax3.set_xlabel('RMS (ppm)')
    ax3.set_ylabel('Frequency')
    ax3.set_title('Distribution of RMS Values per Wavelength')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Bin widths
    ax4 = axes[1, 1]
    bin_widths = np.diff(best_edges)
    bin_centers = (best_edges[:-1] + best_edges[1:]) / 2
    
    ax4.bar(range(len(bin_widths)), bin_widths, alpha=0.7, edgecolor='black')
    ax4.set_xlabel('Bin Number')
    ax4.set_ylabel('Bin Width')
    ax4.set_title(f'Bin Widths (mean: {np.mean(bin_widths):.5f}, std: {np.std(bin_widths):.5f})')
    ax4.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'../../figs/{visit}_robust_optimization_results.png', dpi=300, bbox_inches='tight')
    plt.show()


# Even simpler fallback version
def fallback_even_spacing_optimizer(visit: str = 'F21',
                                  min_bins: int = 16,
                                  max_bins: int = 50,
                                  step_size: int = 2) -> Tuple[np.ndarray, float]:
    """
    Simple optimizer that only tests evenly spaced bins.
    Useful as a fallback when other methods fail.
    """
    
    print(f"Using fallback even-spacing optimizer for {visit}")
    print(f"Testing {min_bins} to {max_bins} bins (step: {step_size})")
    
    # Get wavelength range
    rms_temp, wavelength_range, wavelengths = bin_testing(
        visit=visit,
        bin_edges=np.array([wavelength_range[0], wavelength_range[1]]),
        print_results=False,
        plot=False,
        return_wavelengths=True
    )
    
    wmin, wmax = wavelength_range
    baseline_rms = np.mean(rms_temp)
    
    best_rms = baseline_rms
    best_edges = np.array([wmin, wmax])
    best_n_bins = 1
    
    # Test different numbers of bins
    for n_bins in range(min_bins, max_bins + 1, step_size):
        print(f"Testing {n_bins} bins...", end=' ')
        
        # Evenly spaced bins
        edges = np.linspace(wmin, wmax, n_bins + 1)
        
        # Evaluate
        rms_list = bin_testing(
            visit=visit,
            bin_edges=edges,
            print_results=False,
            plot=False
        )
        avg_rms = np.mean(rms_list)
        
        print(f"RMS: {avg_rms:.2f}")
        
        if avg_rms < best_rms:
            improvement = best_rms - avg_rms
            best_rms = avg_rms
            best_edges = edges
            best_n_bins = n_bins
            print(f"  ↑ New best! Improvement: {improvement:.2f}")
    
    print(f"\nBest: {best_n_bins} bins, RMS: {best_rms:.2f}")
    return best_edges, best_rms


# Main function that tries multiple approaches
def auto_bin_optimizer(visit: str = 'F21',
                      min_bins: int = 16,
                      max_bins: int = 50,
                      try_multiple_methods: bool = True) -> Tuple[np.ndarray, float]:
    """
    Automatic bin optimizer that tries multiple methods.
    """
    
    print(f"Auto bin optimizer for {visit}")
    print(f"Target: {min_bins}-{max_bins} bins")
    
    results = []
    
    # Method 1: Robust optimizer
    if try_multiple_methods:
        print("\n=== Method 1: Robust Optimizer ===")
        try:
            edges1, rms1, _ = robust_bin_optimizer(
                visit=visit,
                min_bins=min_bins,
                max_bins=max_bins,
                plot_progress=False
            )
            results.append(('robust', edges1, rms1))
        except Exception as e:
            print(f"Robust optimizer failed: {e}")
    
    # Method 2: Fallback (even spacing)
    print("\n=== Method 2: Even Spacing ===")
    edges2, rms2 = fallback_even_spacing_optimizer(
        visit=visit,
        min_bins=min_bins,
        max_bins=max_bins,
        step_size=max(1, (max_bins - min_bins) // 10)  # Adaptive step
    )
    results.append(('even_spacing', edges2, rms2))
    
    # Method 3: Hybrid (if we want more options)
    if try_multiple_methods and len(results) > 1:
        print("\n=== Method 3: Best of Both ===")
        # Take the best result from previous methods
        best_method = min(results, key=lambda x: x[2])
        print(f"Best from previous methods: {best_method[0]} with RMS {best_method[2]:.2f}")
        
        # Optional: Try refining it
        edges3 = best_method[1]
        rms3 = best_method[2]
        results.append(('best_of_both', edges3, rms3))
    
    # Select best overall
    best_result = min(results, key=lambda x: x[2])
    
    print(f"\n{'='*60}")
    print(f"FINAL SELECTION")
    print(f"{'='*60}")
    print(f"Best method: {best_result[0]}")
    print(f"Number of bins: {len(best_result[1]) - 1}")
    print(f"RMS: {best_result[2]:.2f}")
    print(f"Bin edges: {best_result[1]}")
    
    return best_result[1], best_result[2]

In [ ]:
warnings.filterwarnings("ignore", category=UserWarning, module="chromatic")

# Option 1: Robust optimizer (handles no-minima case)
best_edges, best_rms, history = robust_bin_optimizer(
    visit='S22',
    min_bins=30,
    max_bins=65,
    plot_progress=True
)

# Example 5: Test your optimized configuration
print("\nTesting optimized configuration...")
final_rms_list = bin_testing(
    visit='S22',
    bin_edges=best_edges,
    print_results=True,
    plot=True
)

In [ ]:
# Option 2: Force even spacing if you want
best_edges, best_rms, history = robust_bin_optimizer(
    visit='S22',
    min_bins=38,
    max_bins=40,
    force_even_spacing=True,  # Force evenly spaced bins
    plot_progress=True
)

# Example 5: Test your optimized configuration
print("\nTesting optimized configuration...")
final_rms_list = bin_testing(
    visit='S22',
    bin_edges=best_edges,
    print_results=True,
    plot=True
)

In [ ]:
# Option 2: Force even spacing if you want
best_edges, best_rms, history = robust_bin_optimizer(
    visit='S22',
    min_bins=50,
    max_bins=63,
    force_even_spacing=True,  # Force evenly spaced bins
    plot_progress=True
)

# Example 5: Test your optimized configuration
print("\nTesting optimized configuration...")
final_rms_list = bin_testing(
    visit='S22',
    bin_edges=best_edges,
    print_results=True,
    plot=True
)

In [ ]:
best_edges, best_rms, history = simple_bin_optimizer(
    visit='S22',
    min_bins=20,
    max_bins=65,
    plot_progress=True
)

# Example 5: Test your optimized configuration
print("\nTesting optimized configuration...")
final_rms_list = bin_testing(
    visit='S22',
    bin_edges=best_edges,
    print_results=True,
    plot=True
)

In [ ]:
_params_config={
    # Ramp Model Parameters
    "r1": (17.0,18.5),
    "r2": (-8.0,-6.5),
    "r3": (-0.0001,0.0001),
    #Breathing params
    "b1":(-0.01,0.01),
    "b2":(-0.01,0.01),
    "b3":(-0.01,0.01),
    "b4":(-0.01,0.01),
    # Planetary Parameters
    "a_rstar": (18.0,18.5),
    "ecc": (0.0,0.2),
    "t0": (-0.001,0.001),
    # Stellar Parameters
    "f_cool_unocculted":(0.2,0.4),
    "spec_scale_factor": (0.95, 1.05),
    "T_unocculted": (2900,3100),
    "T_occulted": (3300,3450),    
    "T_phot": (3800,4100),
    "log_fixedspot_radii":(-1.8,-1.5), #This is the size scale of spots at the active latitudes
    # This spot is occulted
    "spot1_lon": (0.85,0.89),
    "spot1_lat": (1.79,1.83),
    "spot1_rad": (0.05,0.07),
    "spot2_lon":(-0.3,0.5),
    "spot2_lat":(1.4,1.51),
    "spot2_rad":(0.3,0.32),
    #This spot controls the overall rotational modulation
    "spot3_lon": (-1.1,-0.8),
    "spot3_lat": (0.9,1.6),
    "spot3_rad": (0.15,0.25),
}

_priors={
    # Ramp Model Parameters
    "r1": (5.0, 30.0),
    "r2": (-20, -1.0),
    "r3": (-0.01, 0.01),
    #Breathing params
    "b1": (-0.5, 0.5),
    "b2": (-0.5, 0.5),
    "b3": (-0.5, 0.5),
    "b4": (-0.5, 0.5),
    # Transit Parameters
    "a_rstar": (17.0, 21.0),
    "ecc": (0.0,0.2),
    "t0": (-0.001,0.001),
    # Stellar Parameters
    "f_cool_unocculted": (0.1,0.6),
    "spec_scale_factor": (0.5, 1.5),
    "T_unocculted": (2850,3150),
    "T_occulted": (3300,3500),    
    "T_phot": (3800,4150),
    "log_fixedspot_radii": (-3,-1),
    "spot1_lon": (0.8, 0.92),
    "spot1_lat": (1.7, 1.85),
    "spot1_rad": (0.0, 0.15),
    "spot2_lon": (0.0,1.4),
    "spot2_lat": (1.2,1.6),
    "spot2_rad": (0.0,0.5),
    "spot3_lon": (-1.4, 0.0),
    "spot3_lat": (0.8, 1.8),
    "spot3_rad": (0.0, 0.6),
}

In [ ]:
# visit = 'F21'
# fig, ax = plt.subplots(1,1,figsize=(5,3))
# for N_edges in tqdm(list(range(10, 70))):

#     n_edges = int(N_edges)
#     new_bin_edges = np.linspace(1.14,1.64,n_edges) * u.micron
#     _rms = bin_testing(visit='F21',bin_edges = new_bin_edges,
#             print_results = False,plot=False)
#     rms = np.nanstd(_rms)
#     mean_rms = np.nanmean(_rms)
#     max_rms = np.nanmax(_rms)/rms
#     rms_stat = np.nanmin(_rms)/np.nanmean(_rms)

#     ax.scatter(n_edges, rms, color='k',alpha=0.3)

# ax.set_xlabel('N_edges')
# ax.set_ylabel('Std(Residual RMS)')
# ax.set_title('F21')
# plt.savefig(f'../../figs/{visit}_rmsvsbinsize.png',dpi=200)
# plt.show()
# plt.clf()

# visit = 'S22'
# fig, ax = plt.subplots(1,1,figsize=(5,3))
# for N_edges in tqdm(list(range(10, 70))):

#     n_edges = int(N_edges)
#     new_bin_edges = np.linspace(0.82,1.13,n_edges) * u.micron
#     _rms = bin_testing(visit='S22',bin_edges = new_bin_edges,
#             print_results = False,plot=False)
#     rms = np.nanstd(_rms)
#     mean_rms = np.nanmean(_rms)
#     max_rms = np.nanmax(_rms)/rms
#     rms_stat = np.nanmin(_rms)/np.nanmean(_rms)

#     ax.scatter(n_edges, rms,s=float((rms_stat*10)**2),color='k',alpha=0.3)

# ax.set_xlabel('N_edges')
# ax.set_ylabel('Std(Residual RMS)')
# ax.set_title('S22')
# plt.savefig(f'../../figs/{visit}_rmsvsbinsize.png',dpi=200)
# plt.show()
# plt.clf()

In [ ]:
bin_testing(visit='F21',
            bin_edges = F21_bin_edges.value,
            print_results = False,plot=False,return_wavelengths=True)

In [ ]:
'FOR THE TRANSMISSION LIGHT CURVES'

F21_bin_edges = np.linspace(1.14,1.64,49) * u.micron
# for edge in F21_bin_edges:
    # print(f'{edge.value:.6f}')
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges = np.linspace(1.14,1.64,42) * u.micron
# for edge in F21_bin_edges:
    # print(f'{edge.value:.6f}')
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges = np.linspace(1.14,1.64,31) * u.micron
# for edge in F21_bin_edges:
    # print(f'{edge.value:.6f}')
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges = np.linspace(1.14,1.64,24) * u.micron
# for edge in F21_bin_edges:
    # print(f'{edge.value:.6f}' )
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges = np.linspace(1.14,1.64,17) * u.micron
# for edge in F21_bin_edges:
    # print(f'{edge.value:.6f}')
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges =jnp.array([1.14064,1.16381,1.18234,1.20087,
                          1.21477,1.23793,1.25647,1.28426,
                          1.31669,1.33059,1.34449,1.35839,
                          1.39082,1.40935,1.42325,1.43715,
                          1.45568,1.47421,1.51127,1.52980,
                          1.54834,1.57150,1.62246,1.64])*u.micron
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

F21_bin_edges = np.array([1.13, 1.159170,1.210135,
                          1.25183, 1.28426,1.325955,
                          1.34912, 1.390815,1.413980, 
                          1.460305, 1.488105,1.5298,
                          1.566865,1.62245,1.64]) * u.micron
bin_testing(visit='F21',
            bin_edges = F21_bin_edges,
            print_results = True,plot=True)

In [ ]:
S22_bin_edges = np.linspace(0.82,1.13,61) * u.micron
# for edge in S22_bin_edges:
#     print(f'{edge.value:.6f}')
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)

S22_bin_edges = np.linspace(0.82,1.13,53) * u.micron
# for edge in S22_bin_edges:
#     print(f'{edge.value:.6f}')
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)

S22_bin_edges = np.linspace(0.82,1.13,41) * u.micron
# for edge in S22_bin_edges:
#     print(f'{edge.value:.6f}')
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)

S22_bin_edges = np.linspace(0.82,1.13,31) * u.micron
# for edge in S22_bin_edges:
#     print(f'{edge.value:.6f}')
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)

S22_bin_edges = np.linspace(0.82,1.13,22) * u.micron
# for edge in S22_bin_edges:
#     print(f'{edge.value:.6f}')
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)

S22_bin_edges = np.array([0.82, 0.82306047, 0.8304358 ,0.8697709 , 
                          0.87960467,0.88943845, 0.90418911,0.91156444, 
                          0.92139821, 0.93369043, 0.94844109, 0.96073331, 
                          0.96810864, 0.97302553, 0.97794241,0.98531775,
                          1.01481907,1.02465284, 1.03202817, 1.03694506, 
                          1.04432039, 1.05661261, 1.06644638, 1.07382171, 
                          1.10086459,1.11315681,1.12299059, 1.13]) * u.micron
bin_testing(visit='S22',bin_edges = S22_bin_edges,
        print_results = True,plot=True)